In [77]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import joblib

In [78]:
df = pd.read_csv("mfcc_dataset.csv")

X = df.iloc[:, 1:].values.astype(np.float32)
y = df.iloc[:, 0].values

print("Dataset shape:", X.shape)
print("Labels:", np.unique(y))

Dataset shape: (31, 13)
Labels: ['a' 'b' 'c' 'd' 'e' 'hello' 'kukur' 'mero' 'nam' 'namaste' 'noise']


In [79]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

num_classes = len(np.unique(y_encoded))

print("Classes:", label_encoder.classes_)

Classes: ['a' 'b' 'c' 'd' 'e' 'hello' 'kukur' 'mero' 'nam' 'namaste' 'noise']


Normalize Features


In [80]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

Train/Test Split

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [82]:
class MFCCDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [83]:
train_dataset = MFCCDataset(X_train, y_train)
test_dataset = MFCCDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

Neural Network


In [84]:
class MFCCNet(nn.Module):

    def __init__(self, input_size=13, num_classes=3):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.model(x)


model = MFCCNet(input_size=13, num_classes=num_classes)

In [85]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [86]:
epochs = 100

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")

Epoch 1/100 | Loss: 4.7890
Epoch 2/100 | Loss: 4.7549
Epoch 3/100 | Loss: 4.7249
Epoch 4/100 | Loss: 4.6838
Epoch 5/100 | Loss: 4.6288
Epoch 6/100 | Loss: 4.6286
Epoch 7/100 | Loss: 4.5651
Epoch 8/100 | Loss: 4.5250
Epoch 9/100 | Loss: 4.4859
Epoch 10/100 | Loss: 4.4357
Epoch 11/100 | Loss: 4.3812
Epoch 12/100 | Loss: 4.3832
Epoch 13/100 | Loss: 4.2716
Epoch 14/100 | Loss: 4.2919
Epoch 15/100 | Loss: 4.1564
Epoch 16/100 | Loss: 4.1700
Epoch 17/100 | Loss: 4.1246
Epoch 18/100 | Loss: 4.0702
Epoch 19/100 | Loss: 4.0711
Epoch 20/100 | Loss: 3.9048
Epoch 21/100 | Loss: 3.9097
Epoch 22/100 | Loss: 3.8257
Epoch 23/100 | Loss: 3.7797
Epoch 24/100 | Loss: 3.6893
Epoch 25/100 | Loss: 3.6617
Epoch 26/100 | Loss: 3.5478
Epoch 27/100 | Loss: 3.5037
Epoch 28/100 | Loss: 3.5478
Epoch 29/100 | Loss: 3.4049
Epoch 30/100 | Loss: 3.4130
Epoch 31/100 | Loss: 3.2201
Epoch 32/100 | Loss: 3.1371
Epoch 33/100 | Loss: 3.0583
Epoch 34/100 | Loss: 2.9032
Epoch 35/100 | Loss: 2.9653
Epoch 36/100 | Loss: 2.8666
E

In [87]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

print("Accuracy:", correct / total)

Accuracy: 0.42857142857142855


In [88]:
torch.save(model.state_dict(), "mfcc_model.pth")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Model saved successfully.")

Model saved successfully.


In [ ]:
import torch
import numpy as np
import joblib

import DCT  






model = MFCCNet(input_size=13, num_classes=num_classes)
model.load_state_dict(torch.load("mfcc_model.pth"))
model.eval()

scaler = joblib.load("scaler.pkl")
label_encoder = joblib.load("label_encoder.pkl")

predict using dct.py mfcc

In [ ]:
from torchaudio.transforms import MFCC
import melFilterBank
import DCT

def predict_from_dct(startingvalue,framesize):

 
    FFTbins, nextFrame = melFilterBank.selectiveFrequencyBins(
    start=startingvalue,
    framesToCompute=framesize
)

    MFCCS = DCT.get_mfcc(FFTbins)
    
    

   
    sample = MFCCS.mean(axis=0).astype(np.float32)
    # print("Feature vector is:", sample)


    sample = scaler.transform([sample]).astype(np.float32)

  
    sample = torch.tensor(sample, dtype=torch.float32)

   
    with torch.no_grad():
        output = model(sample)
        pred = torch.argmax(output, dim=1).item()

    return label_encoder.inverse_transform([pred])[0]

In [98]:
import importlib, DCT, FFT, melFilterBank; importlib.reload(DCT); importlib.reload(FFT); importlib.reload(melFilterBank)

Total Frames  197
fft bins size 108547


<module 'melFilterBank' from '/home/sas/Desktop/Git_hub/mfcc-speech-recognition/melFilterBank.py'>

In [104]:

currentLabel = None
for i in range(0,FFT.total_frames-47, 10):
    result = predict_from_dct(i,47)
    if(currentLabel != result):
        print(f"Predicted label for frames {i} to {i+47}: {result}")
    currentLabel = result




Predicted label for frames 0 to 47: hello
Predicted label for frames 100 to 147: a
Predicted label for frames 140 to 187: nam


In [93]:
print(list(model.parameters())[0][0][:5])

tensor([-0.1082, -0.2954, -0.1856,  0.3031,  0.2511], grad_fn=<SliceBackward0>)
